In [16]:
# %pip install torch transformers sentence-transformers faiss-cpu pypdf gradio accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 539.2/539.2 MB 46.2 MB/s  0:00:11
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 519.2/519.2 MB 12.2 MB/s  0:00:27
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.4/135.4 MB 18.4 MB/s  0:00:07
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.9/404.9 MB 15.5 MB/s  0:00:23
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 82.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 MB 51.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 67.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.3/218.3 MB 16.9 MB/s  0:00:12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 51.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.9/227.9 MB 18.9 MB/s  0:00:12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.0/151.0 MB 66.9 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.7/285.7 MB 73.6 MB/s  0:00:03
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 108.2 MB/s  0:00:00

In [3]:
policies = [
    [
        "POL-01",
        "Earned Leave",
        "Government employees are credited 30 days of Earned Leave annually, "
        "with half-yearly credits of 15 days on January 1 and July 1."
    ],
    [
        "POL-02",
        "District Office",
        "District Informatics Officers (DIOs) should remain at district "
        "headquarters on working days unless on official tour or approved leave."
    ],
    [
        "POL-03",
        "Data Security",
        "Citizen records containing Personally Identifiable Information (PII) "
        "must remain on approved domestic government cloud infrastructure "
        "with encryption."
    ],
    [
        "POL-04",
        "Casual Leave",
        "Employees may avail Casual Leave subject to approval from the "
        "competent authority and applicable service rules."
    ],
    [
        "POL-05",
        "Office Attendance",
        "Employees must follow prescribed office working hours and maintain "
        "attendance through the authorized attendance management system."
    ],
    [
        "POL-06",
        "Cyber Security",
        "All government systems must use strong passwords, access controls, "
        "and multi-factor authentication wherever applicable."
    ],
    [
        "POL-07",
        "Official Email",
        "Government employees should use authorized official email accounts "
        "for official correspondence and avoid sharing confidential information "
        "through personal email services."
    ],
    [
        "POL-08",
        "Data Backup",
        "Critical government application data should be backed up regularly "
        "according to the approved backup and disaster recovery policy."
    ],
    [
        "POL-09",
        "Software Deployment",
        "Government software applications should undergo functional testing, "
        "security assessment, and approval before production deployment."
    ],
    [
        "POL-10",
        "Official Tour",
        "Official tours must be approved by the competent authority and "
        "documented according to applicable travel regulations."
    ],
    [
        "POL-11",
        "Incident Reporting",
        "Suspected cybersecurity incidents must be reported promptly to "
        "the designated information security team according to established procedures."
    ],
    [
        "POL-12",
        "Asset Management",
        "Government laptops, desktops, and networking equipment must be "
        "recorded in the official asset register and used for authorized purposes."
    ]
]

In [4]:
documents = []

for policy_id, category, content in policies:

    chunk = (
        f"Policy ID: {policy_id}\n"
        f"Category: {category}\n"
        f"Content: {content}"
    )

    documents.append(chunk)

print("Total Chunks:", len(documents))

for i, chunk in enumerate(documents):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk)

Total Chunks: 12

--- Chunk 1 ---
Policy ID: POL-01
Category: Earned Leave
Content: Government employees are credited 30 days of Earned Leave annually, with half-yearly credits of 15 days on January 1 and July 1.

--- Chunk 2 ---
Policy ID: POL-02
Category: District Office
Content: District Informatics Officers (DIOs) should remain at district headquarters on working days unless on official tour or approved leave.

--- Chunk 3 ---
Policy ID: POL-03
Category: Data Security
Content: Citizen records containing Personally Identifiable Information (PII) must remain on approved domestic government cloud infrastructure with encryption.

--- Chunk 4 ---
Policy ID: POL-04
Category: Casual Leave
Content: Employees may avail Casual Leave subject to approval from the competent authority and applicable service rules.

--- Chunk 5 ---
Policy ID: POL-05
Category: Office Attendance
Content: Employees must follow prescribed office working hours and maintain attendance through the authorized attendance 

In [18]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded successfully!


In [19]:
import faiss
import numpy as np

# Generate embeddings from policy documents
embeddings = embedding_model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# Get embedding dimension
dimension = embeddings.shape[1]

# Create FAISS index
index = faiss.IndexFlatIP(dimension)

# Store embeddings
index.add(embeddings)

print("FAISS index created successfully!")
print("Embedding shape:", embeddings.shape)
print("Total indexed chunks:", index.ntotal)

FAISS index created successfully!
Embedding shape: (12, 384)
Total indexed chunks: 12


In [20]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(embeddings)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 12


In [21]:
def retrieve_chunks(query, top_k=3):

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = index.search(
        query_embedding,
        min(top_k, index.ntotal)
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        results.append({
            "policy_id": policies[idx][0],
            "category": policies[idx][1],
            "content": policies[idx][2],
            "score": float(score)
        })

    return results

In [22]:
query = "How many earned leave days are credited in January?"

results = retrieve_chunks(query)

for result in results:
    print("Policy:", result["policy_id"])
    print("Category:", result["category"])
    print("Content:", result["content"])
    print("Similarity:", round(result["score"], 4))
    print("-" * 50)

Policy: POL-01
Category: Earned Leave
Content: Government employees are credited 30 days of Earned Leave annually, with half-yearly credits of 15 days on January 1 and July 1.
Similarity: 0.6641
--------------------------------------------------
Policy: POL-04
Category: Casual Leave
Content: Employees may avail Casual Leave subject to approval from the competent authority and applicable service rules.
Similarity: 0.2785
--------------------------------------------------
Policy: POL-02
Category: District Office
Content: District Informatics Officers (DIOs) should remain at district headquarters on working days unless on official tour or approved leave.
Similarity: 0.2466
--------------------------------------------------


In [23]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

model.eval()

print("Device:", device)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Device: cuda


In [24]:
def rag_chatbot(question):

    results = retrieve_chunks(question, top_k=3)

    context = "\n\n".join(
        f"{r['policy_id']} - {r['category']}: {r['content']}"
        for r in results
    )

    messages = [
        {
            "role": "system",
            "content": (
                "You are a government policy assistant. "
                "Answer using only the provided policy context. "
                "If the answer is unavailable, say that the "
                "information is not available in the policies. "
                "Mention the relevant Policy ID."
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question: {question}"
            )
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response

In [25]:
print(rag_chatbot("How many earned leave days are credited in January?"))

The earned leave days credited in January under the given policies are 15 days.


In [26]:
print(rag_chatbot("Can a DIO leave district headquarters without permission?"))

The given policy does not provide any information about a DIO being allowed to leave district headquarters without permission. Therefore, the answer is:

Not Available in the Policies.
